# EDA · Tanzania · perinatal + ANC detail

**Dataset:** `data/processed/tz_perinatal_anc.csv`: the most recent live birth or stillbirth (7+ months) in the 3 years before the 2022 Tanzania DHS, with antenatal care (ANC) details.
**Outcome:** `perinatal_death` (stillbirth, or death in the first 7 days of life).
**Owner:** Jeff (Tanzania squad)

**Why this notebook matters:** Tanzania is our external validation country. Before we test a Kenyan model here, we need to know what this population looks like, how often the outcome happens, which factors relate to it, and whether any ANC variable behaves in a way that could mislead a model.

**What we cover:**
1. Load and confirm the data matches the proposal
2. Sort the columns into outcome, predictors, exploratory, survey design and leakage
3. Missing data and value ranges
4. The outcome rate (survey-weighted)
5. Derived variables (groups, flags, the checklist)
6. Outcome rate by each background factor
7. Outcome rate by ANC factor, and ANC coverage
8. A leakage check: does ANC reflect how long the pregnancy lasted?
9. The hypertension angle (exploratory)
10. Region (exploratory only)
11. The checklist baseline in this cohort
12. Correlations between predictors
13. Summary: what this means for modelling

**Rule for every rate in this notebook:** use the survey weight (`weight`). DHS deliberately over-samples some areas; unweighted rates describe our sample, not Tanzania.

Clear outputs before committing.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The notebook lives in ml/notebooks/tanzania/, so the data is three folders up.
DATA = Path("../../../data/processed/tz_perinatal_anc.csv")
RANDOM_STATE = 42
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 60)

## 1. Load and confirm the data

We check the row and death counts against the agreed cohort build (recorded in the proposal and in `contracts/cohort_config.yaml`: 5,952 pregnancies, 205 perinatal deaths) and that each pregnancy appears once. This confirms we are all analysing the same file. It is not a check against the DHS report (that is done on the full perinatal cohort) and not a leakage check (section 10).

In [ ]:
df = pd.read_csv(DATA)
print(df.shape)
df.dtypes.value_counts()

In [ ]:
EXPECTED_ROWS, EXPECTED_DEATHS = 5952, 205
assert len(df) == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS} rows, got {len(df)}"
assert df["perinatal_death"].sum() == EXPECTED_DEATHS, "Death count differs from the proposal"
assert not df.duplicated(["caseid", "pidx"]).any(), "Duplicate pregnancies found"
print("Row count, death count and uniqueness confirmed.")

In [ ]:
# Same woman twice, or twins? (genuine rows, not duplicates)
print(
    "Women in file:",
    df["caseid"].nunique(),
    "| women with more than one row:",
    (df["caseid"].value_counts() > 1).sum(),
)
print("Twin rows:", (df["multiple"] == 1).sum())

**Result:** the file matches the agreed cohort (5,952 pregnancies, 205 perinatal deaths) and no pregnancy appears twice. Forty-two women contribute two pregnancies (a recent live birth *and* a recent stillbirth) and twins have one row per baby; these are genuine and kept. Every notebook in the team is analysing the same data.

## 2. Sort the columns

The file has 57 columns, but only some may go into a model. Sorting them now prevents mistakes later:

- **Outcome:** what we predict.
- **Base predictors:** known at booking (history and background); also used in the full-cohort model.
- **ANC predictors:** collected during antenatal care; only available in this cohort.
- **Exploratory only:** `region` (Kenyan counties and Tanzanian regions do not match, so a Kenyan model cannot use it here) and `told_high_bp` (asked at interview, so the diagnosis may come after the pregnancy).
- **Survey design:** `weight` for rates, `v001` (cluster) for splitting data later.
- **Leakage:** known only at or after birth. If a model sees these, it looks brilliant and is useless in a clinic.

The remaining columns are the original DHS codes (`v106`, `m14`, ...) that the readable columns were made from; we use the readable versions.

In [ ]:
OUTCOME = "perinatal_death"
BASE_PREDICTORS = [
    "maternal_age",
    "prior_pregnancies",
    "prior_live_births",
    "prior_stillbirth",
    "prior_loss",
    "preceding_interval_m",
    "multiple",
    "education",
    "wealth_quintile",
    "residence",
]
ANC_PREDICTORS = [
    "anc_visits",
    "first_anc_month",
    "bp_measured",
    "urine_tested",
    "blood_tested",
    "iron_tablets",
]
EXPLORATORY = ["region", "told_high_bp"]  # EDA only, not model inputs
DESIGN = ["weight", "v001"]  # survey weight and cluster
LEAKAGE = [
    "pregnancy_months",
    "birth_weight_g",
    "delivery_place",
    "baby_sex",
    "stillbirth",
    "early_neonatal_death",
    "pregnancy_outcome",
]

for name, cols in [
    ("Outcome", [OUTCOME]),
    ("Base predictors", BASE_PREDICTORS),
    ("ANC predictors", ANC_PREDICTORS),
    ("Exploratory only", EXPLORATORY),
    ("Survey design", DESIGN),
    ("Leakage: never a predictor", LEAKAGE),
]:
    print(f"{name} ({len(cols)}): {', '.join(cols)}")

## 3. Missing data and value ranges

Missing data decides whether we can simply drop incomplete rows or need to impute. We also check *why* a value is missing: a blank birth interval for a first pregnancy is not "unknown"; there was no previous pregnancy. That is structural missingness and must be coded, not imputed.

In [ ]:
cols = BASE_PREDICTORS + ANC_PREDICTORS + EXPLORATORY
missing = pd.DataFrame(
    {
        "missing_n": df[cols].isna().sum(),
        "missing_%": df[cols].isna().mean() * 100,
    }
).sort_values("missing_%", ascending=False)
missing

**Result:** missing data is negligible: under 0.5% in every predictor except the birth interval. `told_high_bp` and `region` are complete in Tanzania.

The birth interval is missing for 20.5% of pregnancies (1,217), which looks serious but is not; the next cell shows why.

In [ ]:
# Is the missing birth interval explained by first pregnancies?
first = df["prior_pregnancies"] == 0
print(
    "Missing interval among first pregnancies:",
    df.loc[first, "preceding_interval_m"].isna().sum(),
    "of",
    first.sum(),
)
print(
    "Missing interval among later pregnancies:", df.loc[~first, "preceding_interval_m"].isna().sum()
)

**Result:** 1,208 of the 1,217 missing intervals are first pregnancies, which have no previous pregnancy to measure from. Only 9 later pregnancies are genuinely missing.

**Decision:** do not impute the interval. Use `first_pregnancy` as its own category (as in `interval_group`), so the model treats "first pregnancy" as information, not a gap. The 9 genuinely missing rows can be dropped.

Now the value ranges. We look for clinically impossible or suspicious values (very young mothers, more than 20 ANC visits, a first visit after month 9, a gap between pregnancies shorter than a pregnancy).

In [ ]:
numeric = [
    "maternal_age",
    "prior_pregnancies",
    "prior_live_births",
    "preceding_interval_m",
    "anc_visits",
    "first_anc_month",
]
df[numeric].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T

In [ ]:
print("Mothers under 15:", (df["maternal_age"] < 15).sum())
print("More than 20 ANC visits:", (df["anc_visits"] > 20).sum())
print("First visit after month 9:", (df["first_anc_month"] > 9).sum())
print("Interval under 9 months:", (df["preceding_interval_m"] < 9).sum())

**Result:** values are clinically plausible. Mothers are 13 to 48 years old (median 27); ANC visits range from 0 to 16 (median 4); first visits are at month 0 to 9.

Points to note:
- 8 mothers were under 15 at the end of the pregnancy: plausible (very young mothers) but worth flagging; kept.
- 1 interval under 9 months: likely a recall or dating error; too few to matter.
- No impossible ANC values (none above 20 visits, none after month 9).

## 4. Helper functions

Three small functions used throughout:

- `weighted_rate`: the survey-weighted rate per 1,000.
- `rate_ci`: the same rate with an approximate 95% confidence interval. Because the outcome is rare, many groups have few deaths, and without an interval a big-looking difference may be noise. The interval uses the weighted rate and the "effective sample size" (how many unweighted women the weighted sample is worth). It ignores clustering, so it is slightly too narrow; proper survey-design intervals come in the modelling stage.
- `rate_by`: the rate within each level of a variable, with counts, the interval, and a flag for groups with fewer than 30 deaths (unstable).

In [ ]:
def weighted_rate(d, outcome=OUTCOME, per=1000):
    """Survey-weighted rate per 1,000 (or per 100 with per=100)."""
    return per * np.average(d[outcome], weights=d["weight"])


def rate_ci(d, outcome=OUTCOME, per=1000):
    """Weighted rate with an approximate 95% CI (Wilson, Kish effective sample size)."""
    w, y = d["weight"], d[outcome]
    p = np.average(y, weights=w)
    n_eff = w.sum() ** 2 / (w**2).sum()
    z = 1.96
    centre = (p + z**2 / (2 * n_eff)) / (1 + z**2 / n_eff)
    half = z * np.sqrt(p * (1 - p) / n_eff + z**2 / (4 * n_eff**2)) / (1 + z**2 / n_eff)
    return per * p, per * max(centre - half, 0), per * (centre + half)


def rate_by(d, var, outcome=OUTCOME, order=None):
    """Weighted outcome rate per 1,000 within each level of `var`."""
    rows = []
    for level, g in d.dropna(subset=[var]).groupby(var, observed=True):
        rate, lo, hi = rate_ci(g, outcome)
        rows.append(
            {
                "level": level,
                "n": len(g),
                "deaths": int(g[outcome].sum()),
                "rate_per_1000": rate,
                "ci_low": lo,
                "ci_high": hi,
                "few_events": int(g[outcome].sum()) < 30,
            }
        )
    out = pd.DataFrame(rows).set_index("level")
    return out.reindex(order) if order is not None else out

## 5. The outcome

Perinatal death is rare. The weighted rate per 1,000 is the figure that describes Tanzania; its rarity is why the models will be judged on PR-AUC, not accuracy (predicting "survives" for everyone would be about 96.5% accurate and useless).

In [ ]:
overall = pd.DataFrame(
    {
        "events": [
            int(df["stillbirth"].sum()),
            int(df["early_neonatal_death"].sum()),
            int(df[OUTCOME].sum()),
        ],
        "unweighted_per_1000": [
            1000 * df["stillbirth"].mean(),
            1000 * df["early_neonatal_death"].mean(),
            1000 * df[OUTCOME].mean(),
        ],
        "weighted_per_1000": [
            weighted_rate(df, "stillbirth"),
            weighted_rate(df, "early_neonatal_death"),
            weighted_rate(df),
        ],
    },
    index=["stillbirth", "early_neonatal_death", "perinatal_death"],
)
overall

**Result:** weighted perinatal mortality is **35.2 per 1,000** (21.8 stillbirths and 13.4 early neonatal deaths). Stillbirths make up about 60% of perinatal deaths, so a model that only looked at newborn factors would miss most of the problem.

Weighted and unweighted rates are close (35.2 vs 34.4), but we still report weighted rates throughout.

At about 3.5%, this is a rare outcome: models will be judged on PR-AUC and trained with class weights, not judged on accuracy.

## 6. Derived variables

We create the groupings clinicians think in, and the flags the model and the checklist need:

- **Age, parity and birth-interval groups**, matching the checklist cut-offs (under 18, 35+, 5+ births, under 24 months).
- **`first_pregnancy` and `short_interval`**: the correct way to handle the missing interval for first pregnancies.
- **ANC groups**: no ANC, 1 to 3, 4 to 7, 8+ visits (WHO now recommends at least 8 contacts); trimester of first visit.
- **`checklist_flag`**: the current risk checklist: any one of the five items flags the pregnancy as high risk. This is the baseline our models must beat.

In [ ]:
df["first_pregnancy"] = (df["prior_pregnancies"] == 0).astype(int)
df["short_interval"] = (df["preceding_interval_m"] < 24).astype(int)

df["age_group"] = pd.cut(
    df["maternal_age"], [0, 17, 24, 34, 60], labels=["<18", "18-24", "25-34", "35+"]
)
df["parity_group"] = pd.cut(
    df["prior_live_births"], [-1, 0, 2, 4, 30], labels=["0", "1-2", "3-4", "5+"]
)
df["interval_group"] = np.select(
    [df["first_pregnancy"] == 1, df["preceding_interval_m"] < 24, df["preceding_interval_m"] < 36],
    ["First pregnancy", "<24 months", "24-35 months"],
    default="36+ months",
)

df["no_anc"] = (df["anc_visits"] == 0).astype(int)
df["anc4plus"] = (df["anc_visits"] >= 4).astype(int)
df["anc_group"] = pd.cut(df["anc_visits"], [-1, 0, 3, 7, 40], labels=["None", "1-3", "4-7", "8+"])
df["first_anc_trimester"] = pd.cut(
    df["first_anc_month"],
    [-1, 0, 3, 6, 10],
    labels=["No ANC", "1st (0-3 m)", "2nd (4-6 m)", "3rd (7+ m)"],
)

df["checklist_flag"] = (
    ((df["maternal_age"] < 18) | (df["maternal_age"] >= 35))
    | (df["prior_live_births"] >= 5)
    | (df["preceding_interval_m"] < 24)
    | (df["multiple"] == 1)
    | (df["prior_stillbirth"] == 1)
).astype(int)

df[["first_pregnancy", "short_interval", "no_anc", "anc4plus", "checklist_flag"]].mean().mul(
    100
).round(1)

**Result:** 20% of pregnancies are first pregnancies; 16% followed an interval under 24 months; 11% had no ANC at all; 65% had four or more visits.

The current checklist flags **44%** of pregnancies as high risk: nearly half of all women. A tool that flags half the population gives clinics little help deciding who needs extra attention.

## 7. Distributions by outcome

Do the deaths look different from the survivors on the continuous variables? Histograms are scaled to the share of each group so the 205 deaths can be compared with the 5,747 survivors.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, bins in zip(axes, ["maternal_age", "anc_visits", "first_anc_month"], [30, 20, 11]):
    for value, label in [(0, "Survived"), (1, "Perinatal death")]:
        ax.hist(
            df.loc[df[OUTCOME] == value, col].dropna(),
            bins=bins,
            density=True,
            alpha=0.5,
            label=label,
        )
    ax.set_title(col)
    ax.set_ylabel("Share of group")
axes[0].legend()
plt.suptitle("Tanzania ANC cohort: distributions by outcome (unweighted)")
plt.tight_layout()
plt.show()

**Result:** the deaths lean towards older mothers: a larger share of deaths than survivors are in mothers aged 30 to 46. For ANC, the deaths have more women with no visits or only 1 to 3, and fewer with 5 to 7. The month of the first visit looks broadly similar between the two groups.

These are unweighted shapes; the weighted rates by group follow.

## 8. Perinatal mortality by background factor

For each factor known at booking: the weighted death rate per 1,000 in each group, with its 95% interval. This tells us which factors carry signal for the model, and whether Tanzania shows the patterns clinicians expect (for example, a previous stillbirth raising risk).

In [ ]:
BASE_GROUPS = {
    "age_group": ["<18", "18-24", "25-34", "35+"],
    "parity_group": ["0", "1-2", "3-4", "5+"],
    "interval_group": ["First pregnancy", "<24 months", "24-35 months", "36+ months"],
    "prior_stillbirth": [0, 1],
    "prior_loss": [0, 1],
    "multiple": [0, 1],
    "education": ["No education", "Primary", "Secondary", "Higher"],
    "wealth_quintile": ["Poorest", "Poorer", "Middle", "Richer", "Richest"],
    "residence": ["Urban", "Rural"],
}
base_tables = {var: rate_by(df, var, order=order) for var, order in BASE_GROUPS.items()}
pd.concat(base_tables, names=["variable", "level"])

In [ ]:
def plot_rates(tables, title, ncols=3):
    n = len(tables)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.6 * nrows))
    axes = np.atleast_1d(axes).ravel()
    overall_rate = weighted_rate(df)
    for ax, (var, t) in zip(axes, tables.items()):
        t = t.dropna(subset=["rate_per_1000"])
        x = np.arange(len(t))
        err = [t["rate_per_1000"] - t["ci_low"], t["ci_high"] - t["rate_per_1000"]]
        colours = ["lightgrey" if few else "steelblue" for few in t["few_events"]]
        ax.bar(x, t["rate_per_1000"], yerr=err, capsize=3, color=colours)
        ax.axhline(overall_rate, ls="--", color="black", lw=1)
        ax.set_xticks(x, [str(i) for i in t.index], rotation=30)
        ax.set_title(var)
        ax.set_ylabel("Deaths per 1,000")
    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle(title + "\n(dashed = overall rate; grey = fewer than 30 deaths; bars = 95% CI)")
    plt.tight_layout()
    plt.show()


plot_rates(base_tables, "Tanzania ANC cohort: weighted perinatal mortality by background factor")

**Result:** the classic obstetric risk factors show the strongest signal (weighted deaths per 1,000):

| Factor | Higher-risk group | Comparison |
|---|---|---|
| Twins | 146 | 33 (singleton) |
| Previous stillbirth | 120 | 32 |
| Any previous loss | 69 | 31 |
| Mother 35+ | 52 | 31 to 32 (18 to 34) |

- **Parity is U-shaped:** first births (49) and 5+ previous births (48) carry more risk than 1 to 4 (24 to 28). A single straight-line term would miss this; use groups or a tree-based model.
- **Birth interval:** lowest risk at 24 to 35 months (22); higher for first pregnancies (43) and under 24 months (41).
- **Teenage mothers** (<18) show 30, but with only 7 deaths the interval is very wide; we cannot conclude anything.

**Unexpected:** richer, urban and more educated women show *higher* rates (Richest 53 vs Poorest 27; Urban 44 vs Rural 32). This runs against what clinicians would expect. The likely explanation is reporting: women who deliver in facilities are more likely to have a stillbirth or early death recognised and remembered as such. We should not read this as poverty being protective, and should discuss as a team whether wealth and residence belong in the model.

## 9. Perinatal mortality by antenatal care

This is what makes this cohort different from the headline one. We look at the death rate by number of visits, timing of the first visit, and whether BP, urine and blood were checked and iron given; then at how common each of these is (coverage).

Read these carefully: ANC is not randomly assigned. Women who already have problems may book earlier and attend more, which can make "good" ANC look harmful.

In [ ]:
ANC_GROUPS = {
    "anc_group": ["None", "1-3", "4-7", "8+"],
    "first_anc_trimester": ["No ANC", "1st (0-3 m)", "2nd (4-6 m)", "3rd (7+ m)"],
    "bp_measured": [0, 1],
    "urine_tested": [0, 1],
    "blood_tested": [0, 1],
    "iron_tablets": [0, 1],
}
anc_tables = {var: rate_by(df, var, order=order) for var, order in ANC_GROUPS.items()}
plot_rates(anc_tables, "Tanzania ANC cohort: weighted perinatal mortality by antenatal care")
pd.concat(anc_tables, names=["variable", "level"])

**Result:** antenatal care shows weaker and more puzzling patterns than the obstetric history:

- **Number of visits:** a gentle fall in risk with more visits (none 42, 1 to 3 40, 4 to 7 33, 8+ 29), but the intervals overlap and the extreme groups have fewer than 30 deaths.
- **First visit:** women who booked in the **1st trimester have the highest rate** (45) and those booking in the 2nd the lowest (28). This is probably reverse causation: women who already know of a problem (a previous loss, illness) book early.
- **BP measured, urine tested, blood tested:** no meaningful difference (for example BP measured 36 vs not measured 33).
- **Iron tablets:** slightly lower with iron (34 vs 40), intervals overlap.

ANC variables are therefore weak, possibly misleading, predictors on their own. They should be tested in the model, but we should not expect them to add much over obstetric history.

In [ ]:
coverage = pd.Series(
    {
        "No ANC at all": weighted_rate(df, "no_anc", per=100),
        "4+ visits": weighted_rate(df, "anc4plus", per=100),
        "First visit in 1st trimester": 100
        * np.average(df["first_anc_month"].between(1, 3), weights=df["weight"]),
        "BP measured": weighted_rate(df.dropna(subset=["bp_measured"]), "bp_measured", per=100),
        "Urine tested": weighted_rate(df.dropna(subset=["urine_tested"]), "urine_tested", per=100),
        "Blood tested": weighted_rate(df.dropna(subset=["blood_tested"]), "blood_tested", per=100),
        "Iron tablets": weighted_rate(df.dropna(subset=["iron_tablets"]), "iron_tablets", per=100),
    },
    name="weighted %",
)
coverage.to_frame()

**Result (weighted):** 10% of Tanzanian women had no ANC; 65% had 4+ visits; only 34% booked in the first trimester. BP was measured for 69% and urine tested for 63%, the two checks that detect hypertension in pregnancy. Blood tests and iron reached 82%.

So roughly **one in three women never had her BP checked**, despite it being the cheapest way to detect pre-eclampsia. This gap is a key link to our qualitative work on hypertension care.

## 10. Leakage check: does ANC reflect how long the pregnancy lasted?

A pregnancy that ends at 7 months has had less time for ANC visits than one that reaches 9 months, and early-ending pregnancies are exactly where most deaths are. If so, "few visits" partly measures "short pregnancy", which is only known at the end. We check visits and deaths by pregnancy length. (`pregnancy_months` is used here only to check; it never goes into a model.)

In [ ]:
trunc = df.groupby("pregnancy_months").agg(
    n=("anc_visits", "size"),
    mean_anc_visits=("anc_visits", "mean"),
    deaths=(OUTCOME, "sum"),
)
trunc["death_rate_per_1000"] = 1000 * trunc["deaths"] / trunc["n"]
trunc

**Result:** a clear leakage risk.

| Pregnancy lasted | Mean ANC visits | Deaths per 1,000 |
|---|---|---|
| 7 months | 3.3 | 582 |
| 8 months | 3.6 | 346 |
| 9 months | 4.0 | 25 |

Pregnancies that ended early had fewer ANC visits *and* far more deaths. Part of the link between "few visits" and death is therefore just "the pregnancy ended early", which is only known at the end. A model using the raw visit count would partly be using information from the future.

**Recommendation:** prefer `anc4plus` and `first_anc_month` over `anc_visits`, and run a sensitivity analysis on 9-month pregnancies only.

## 11. The hypertension angle (exploratory)

Hypertension in pregnancy is central to our qualitative work but not recorded directly in DHS. Here we look at the two proxies: whether BP was measured at ANC, and whether the woman was ever told she has high BP. Tanzania asked the second question of all women (Kenya only of a subsample). Exploratory only, because the "told" answer may refer to a diagnosis after this pregnancy.

In [ ]:
print("told_high_bp answered by", f"{df['told_high_bp'].notna().mean():.0%}", "of women")
htn = pd.concat(
    {
        "told_high_bp": rate_by(df, "told_high_bp", order=[0, 1]),
        "bp_measured": rate_by(df, "bp_measured", order=[0, 1]),
    },
    names=["variable", "level"],
)
htn

**Result:** women ever told they had high BP had more than double the perinatal mortality (80 vs 33 per 1,000; 354 women, 30 deaths). This is the strongest single association with hypertension in the data, and fits the clinical picture of pre-eclampsia causing stillbirth.

It stays exploratory: the question was asked at interview, so the diagnosis may have come after this pregnancy. BP being *measured* at ANC shows no difference (36 vs 33), because a measurement on its own does not change outcomes unless high readings lead to action.

In [ ]:
# Were the women told they had high BP mostly those whose BP was measured?
pd.crosstab(df["bp_measured"], df["told_high_bp"], margins=True)

**Result:** 320 of the 354 women told they had high BP (90%) had their BP measured at ANC. Diagnosis depends on being measured: among the 1,726 women never measured, only 34 were ever told. Undiagnosed hypertension is likely hiding in the unmeasured group, so `told_high_bp` underestimates true hypertension.

## 12. Region (exploratory only)

Regional differences are useful context for the report, but region cannot be a model input: Kenyan counties do not exist in Tanzania, so a Kenyan model could not score a Tanzanian woman. Most regions have fewer than 30 deaths, so their rates are unstable.

In [ ]:
region = rate_by(df, "region").sort_values("rate_per_1000", ascending=False)
region.head(10)

**Result:** the highest regional rates are in Kusini Unguja (68), Dar es Salaam (64) and Iringa (58 per 1,000), but every region has fewer than 30 deaths and the intervals are very wide. Regional differences are useful context for the report but are not reliable enough to rank regions, and region is excluded from the model (it does not exist in Kenya).

## 13. The checklist baseline in this cohort

How does the current checklist perform here? Sensitivity = the share of deaths it flagged in advance; PPV = the share of flagged women whose baby died. These are the numbers our models must beat at the same workload. (Unweighted counts, as in a screening table; the modelling notebook repeats this on the test set.)

In [ ]:
def screening_summary(flag, y):
    tp = ((flag == 1) & (y == 1)).sum()
    fp = ((flag == 1) & (y == 0)).sum()
    fn = ((flag == 0) & (y == 1)).sum()
    tn = ((flag == 0) & (y == 0)).sum()
    return pd.Series(
        {
            "flagged_%": 100 * flag.mean(),
            "sensitivity_%": 100 * tp / (tp + fn),
            "specificity_%": 100 * tn / (tn + fp),
            "PPV_%": 100 * tp / (tp + fp),
            "NPV_%": 100 * tn / (tn + fn),
        }
    )


screening_summary(df["checklist_flag"], df[OUTCOME]).to_frame("checklist (Tanzania ANC cohort)")

**Result:** in this cohort the checklist flags 44% of pregnancies but catches only **54% of deaths** (specificity 57%). Only **4.2%** of flagged women lose their baby; that is barely more than the 3.4% overall rate. Among women it does *not* flag, 2.8% still lose their baby (NPV 97.2%).

The checklist is therefore a weak screen: it flags many women and misses nearly half the deaths. This is the bar our models must clear.

## 14. Correlations between predictors

Strongly correlated predictors carry the same information twice. That can make logistic regression coefficients unstable and SHAP explanations confusing. We list any pair above 0.7 (Spearman, which handles counts and 0/1 flags).

In [ ]:
corr_cols = [
    "maternal_age",
    "prior_pregnancies",
    "prior_live_births",
    "prior_loss",
    "prior_stillbirth",
    "first_pregnancy",
    "short_interval",
    "multiple",
    "anc_visits",
    "first_anc_month",
    "bp_measured",
    "urine_tested",
    "blood_tested",
    "iron_tablets",
    OUTCOME,
]
corr = df[corr_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)), corr_cols, rotation=90)
ax.set_yticks(range(len(corr_cols)), corr_cols)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im)
ax.set_title("Spearman correlations, Tanzania ANC cohort")
plt.tight_layout()
plt.show()

high = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
high[high.abs() > 0.7].sort_values(ascending=False)

**Result:** two pairs are highly correlated:
- `prior_pregnancies` and `prior_live_births` (0.98): almost the same information. **Keep `prior_live_births`** (it matches the checklist) and drop `prior_pregnancies` from the model.
- `maternal_age` with parity (0.78): expected (older women have had more children). Fine for tree models; watch for unstable coefficients in logistic regression.

`prior_pregnancies` and `first_pregnancy` correlate at −0.71 by construction. No ANC variable is strongly correlated with any other predictor, and no single predictor is strongly correlated with the outcome, which is typical for a rare outcome and is why combining factors in a model may help.

## 15. Summary: what this means for modelling

**Data quality**
- 5,952 pregnancies, 205 perinatal deaths (128 stillbirths, 77 early neonatal deaths): matches the proposal.
- Missing data under 0.5% in every predictor except the birth interval (20.5%), and that is entirely first pregnancies (1,208 of 1,217). **Decision:** use `first_pregnancy` plus interval groups; do not impute the interval.
- 8 mothers under 15; no impossible ANC values.

**Outcome**
- Weighted perinatal mortality 35.2 per 1,000 (stillbirth 21.8, early neonatal 13.4). About 3.5% of pregnancies: a rare outcome, so PR-AUC and class weights, not accuracy.

**Strongest signals (weighted deaths per 1,000)**
- Twins 146 vs 33; previous stillbirth 120 vs 32; any previous loss 69 vs 31; mother 35+ 52 vs about 31 to 32.
- Parity is U-shaped: first births (49) and 5+ births (48) are higher than 1 to 4 (24 to 28). A straight-line term would miss this, so use groups or a tree model.
- Birth interval: lowest at 24 to 35 months (22).

**Results to interpret with care**
- Richer, urban and more educated women show *higher* rates (Richest 53 vs Poorest 27; Urban 44 vs Rural 32). Most likely better recognition and reporting of stillbirths and early deaths (facility births), not a true protective effect of poverty. Discuss with the team before using wealth and residence as predictors.
- Booking in the 1st trimester shows a *higher* rate (45) than the 2nd (28): likely women with known problems booking early (reverse causation).
- BP measured at ANC shows no protective difference (36 vs 33).

**Leakage warning: ANC visits**
- Pregnancies ending at 7 months averaged 3.3 visits with 582 deaths per 1,000; at 9 months, 4.0 visits and 25 per 1,000. Fewer visits partly reflects a shorter pregnancy, which is only known at the end. **Recommendation:** prefer `anc4plus` and `first_anc_month` over the raw visit count, and run a sensitivity analysis restricted to 9-month pregnancies.

**Hypertension (exploratory)**
- Told high BP: 80 vs 33 per 1,000 (354 women, 30 deaths). A strong association, but timing is uncertain; keep it for the hypertension analysis, not the main model. 320 of the 354 women told they had high BP had their BP measured at ANC.

**Checklist baseline (this cohort)**
- Flags 44% of pregnancies, catches 54% of deaths, PPV 4.2%. Our model must beat this.

**Correlations**
- `prior_pregnancies` and `prior_live_births` correlate at 0.98: keep one (suggest `prior_live_births`, matching the checklist). Maternal age correlates 0.78 with parity; fine for tree models, watch in logistic regression.